<a href="https://colab.research.google.com/github/murtaza-x/flyrank-repo/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/murtaza-x/flyrank-repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)


**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

**Task type:** Ranking / Scoring

The goal is to score and prioritize existing content based on its opportunity for a refresh. The output should help a content/editorial team decide which content should be reviewed and improved first.

This is a ranking/scoring problem because the main decision is not simply whether a page is good or bad. The goal is to produce a priority score that allows content items to be ordered from higher refresh opportunity to lower refresh opportunity.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/murtaza-x/flyrank-repo/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

# Create a sketch of the proxy target column

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "declining"
).astype(int)

df[[
    "content_id",
    "trend_direction",
    "is_declining_label"
]].head(10)

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,0
1,content_a1fb4e703a9e,down,0
2,content_9aa793d4d895,down,0
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,0
5,content_d4084a4bc775,down,0
6,content_9a34b442b552,down,0
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,0
9,content_c27558df2b0c,down,0


## 3. Success metric

**Success metric: Precision@K**

The main goal is to help a content team decide which pages to review first. Therefore, Precision@K is the most useful metric because it measures how many of the top K pages recommended by the system are actually positive according to the chosen proxy.

For example, if the team can review only 50 pages, Precision@50 tells us what proportion of those 50 recommended pages were correctly identified as declining.

This metric matches the real action because the output is a ranked review queue, not just a prediction for every page.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Chosen success metric: Precision@K")
print("Example operational metric: Precision@50")

Chosen success metric: Precision@K
Example operational metric: Precision@50


## 4. The unit of analysis, as a real dataframe
**Unit of analysis: One row = one pseudonymized content item.**

Each row represents one content item belonging to a pseudonymized client. The row contains measurements about that content item, such as impressions, sessions, content age, freshness, and trend information.

The decision supported by the model is therefore made at the content-item level: each content item receives evidence or a score that can be used to decide whether it should be reviewed earlier than other items.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unit of analysis as a real dataframe

unit_of_analysis = df[[
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
    "is_declining_label"
]]

print("One row = one pseudonymized content item")
print("Shape:", unit_of_analysis.shape)

unit_of_analysis.head(10)

One row = one pseudonymized content item
Shape: (30000, 7)


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,187,down,0
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down,0
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down,0
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down,0
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down,0
6,content_9a34b442b552,client_8722616204,20,1,90,down,0
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down,0
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down,0


## 5. Why ML beats a fixed rule here

A fixed rule could identify simple cases, for example: "review a page if it is old and has high impressions." However, refresh opportunity depends on several signals that can interact with each other.

For example, impressions, sessions, content age, freshness, average position, CTR, engagement, scroll rate, and content characteristics may provide different evidence about whether a page deserves attention.

ML can learn patterns across multiple signals and produce a probability or score that can be used to rank content items. This is more flexible than relying on one manually chosen threshold.

The output does not guarantee that refreshing a page will improve it. Instead, it supports a content/editorial team in deciding which pages to review first when time and resources are limited.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Example of why one fixed rule is limited

fixed_rule_example = (
    (df["content_age_days"] >= 180) &
    (df["impressions_90d"] >= 500)
)

print(
    "Pages selected by a simple age + impressions rule:",
    fixed_rule_example.sum()
)
print(
    "Total pages:",
    len(df)
)

Pages selected by a simple age + impressions rule: 9929
Total pages: 30000


## Self-check

Before you submit, confirm each line honestly:

- [ +] Every section above is filled — markdown thinking AND the code that backs it
- [+ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+ ] No client names, URLs, or private queries anywhere
- [+ ] My claims use careful words: observed, measured, directional, decision-support
- [ +] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.